# Machine Learning Model Comparison for IoT Intrusion Detection

This notebook evaluates multiple supervised machine learning algorithms
(Random Forest, SVM, KNN, Decision Tree, Logistic Regression)
on an IoT intrusion detection dataset generated using IoT-Flock.


## 1. Imports and configuration

In [ ]:
import pandas as pd
import ipaddress
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression


## 2. Dataset Loading

In [ ]:
df = pd.read_csv("balanced_dataset.csv")
df.head()


## 3. Data Preprocessing

In [ ]:
# Convert IP addresses to integers
df['ip.src'] = df['ip.src'].apply(lambda x: int(ipaddress.IPv4Address(x)))
df['ip.dst'] = df['ip.dst'].apply(lambda x: int(ipaddress.IPv4Address(x)))

# Check missing values
df.isnull().sum()


## 4. Label Distribution

In [ ]:
df['label'].value_counts()


## 5. Feature / Target Split

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

print(X.shape, y.shape)


## 6. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## 7. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


## 8. model Definitions

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}


## 9. Training and Evaluation Function

In [ ]:
def evaluate_model(y_test, y_pred):
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted")
    }


## 10. Model Comparison Loop

In [ ]:
results = []
training_times = []

for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    training_times.append(time.time() - start)

    y_pred = model.predict(X_test)
    metrics = evaluate_model(y_test, y_pred)
    metrics["model"] = name
    results.append(metrics)


## 11. Results Table

In [ ]:
results_df = pd.DataFrame(results)
results_df


## 12. Accuracy Comparison plot

In [ ]:
plt.figure(figsize=(10,5))
sns.barplot(data=results_df, x="model", y="accuracy")
plt.title("Model Accuracy Comparison")
plt.xticks(rotation=45)
plt.show()


## 13. Confusion Matrix (One Best Model)

In [ ]:
best_model = models["Random Forest"]
y_pred = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Random Forest")
plt.show()
